# 04_omniscape.ipynb

## Ecological Connectivity Modeling using Omniscape

This notebook runs circuit-theory-based connectivity modeling using resistance surfaces.

### Inputs
- Species resistance GeoTIFF (EPSG:3035)

### Outputs
- Current flow maps
- Flow potential
- Normalized connectivity surfaces
- QGIS-ready GeoTIFF outputs


## Imports

In [1]:
from pathlib import Path
import subprocess
import rasterio
import numpy as np
import matplotlib.pyplot as plt


## Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

SPECIES = 'Emys_orbicularis'

RESISTANCE_RASTER = PROJECT_ROOT / 'data' / 'processed' / 'resistance' / f'{SPECIES}_resistance.tif'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'connectivity' / SPECIES
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(RESISTANCE_RASTER)


/home/linda/Documents/myData/data-management/data/processed/resistance/Emys_orbicularis_resistance.tif


## Check Resistance Raster

In [3]:
with rasterio.open(RESISTANCE_RASTER) as src:
    resistance = src.read(1)
    profile = src.profile

print(resistance.shape)
print(profile['crs'])


(543, 292)
EPSG:3035


## Omniscape Configuration

We use Omniscape.jl via subprocess for stability.

In [4]:
OMNISCAPE_JL = PROJECT_ROOT / 'env' / 'run_omniscape.jl'

print(OMNISCAPE_JL)


/home/linda/Documents/myData/data-management/env/run_omniscape.jl


## Create Minimal Omniscape Julia Script (if missing)

In [5]:
if not OMNISCAPE_JL.exists():
    OMNISCAPE_JL.write_text('''
using Omniscape

input_raster = ARGS[1]
output_dir = ARGS[2]

run_omniscape(input_raster;
    outdir=output_dir,
    block_size=50,
    radius=10,
    project_name="eco_connectivity"
)
''')

print('Julia script ready')


Julia script ready


## Run Omniscape

In [6]:
# NOTE: requires Julia + Omniscape.jl installed

cmd = [
    'julia',
    str(OMNISCAPE_JL),
    str(RESISTANCE_RASTER),
    str(OUTPUT_DIR)
]

print('Running Omniscape...')
print(' '.join(cmd))

# Uncomment when ready
# subprocess.run(cmd, check=True)


Running Omniscape...
julia /home/linda/Documents/myData/data-management/env/run_omniscape.jl /home/linda/Documents/myData/data-management/data/processed/resistance/Emys_orbicularis_resistance.tif /home/linda/Documents/myData/data-management/data/processed/connectivity/Emys_orbicularis


## Expected Outputs

In [7]:
print('Expected outputs in:', OUTPUT_DIR)

expected = [
    'current_flow.tif',
    'flow_potential.tif',
    'normalized_current.tif'
]

for f in expected:
    print('-', f)


Expected outputs in: /home/linda/Documents/myData/data-management/data/processed/connectivity/Emys_orbicularis
- current_flow.tif
- flow_potential.tif
- normalized_current.tif


## Quick Visualization (if output exists)

In [8]:
flow_file = OUTPUT_DIR / 'current_flow.tif'

if flow_file.exists():
    with rasterio.open(flow_file) as src:
        flow = src.read(1)

    plt.figure(figsize=(10, 8))
    plt.imshow(flow, cmap='magma')
    plt.colorbar(label='Current Flow')
    plt.title(f'{SPECIES} Connectivity (Omniscape)')
    plt.show()
else:
    print('Run Omniscape first.')


Run Omniscape first.


## Summary

In [9]:
print('--- CONNECTIVITY PIPELINE ---')
print('Species:', SPECIES)
print('Resistance input:', RESISTANCE_RASTER)
print('Output dir:', OUTPUT_DIR)
print('Model: circuit-theory connectivity (Omniscape)')


--- CONNECTIVITY PIPELINE ---
Species: Emys_orbicularis
Resistance input: /home/linda/Documents/myData/data-management/data/processed/resistance/Emys_orbicularis_resistance.tif
Output dir: /home/linda/Documents/myData/data-management/data/processed/connectivity/Emys_orbicularis
Model: circuit-theory connectivity (Omniscape)
